# Métricas de evaluación

Entrena un modelo regularizado reproducible, selecciona pesos con el conjunto de desarrollo y calcula métricas finales una sola vez sobre test.


# Preparación de datos MNIST

Este bloque se incluye en cada notebook para que pueda ejecutarse de forma independiente.

In [ ]:
# Librerías, semilla y carga de MNIST
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_dev, y_train, y_dev = train_test_split(
    x_train_full, y_train_full, test_size=0.20, random_state=SEED, stratify=y_train_full
)
x_train = x_train.astype("float32") / 255.0
x_dev = x_dev.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print("Train:", x_train.shape, "Dev:", x_dev.shape, "Test:", x_test.shape)


## Entrenamiento del modelo evaluado

El conjunto de prueba se reserva para la evaluación final.

In [ ]:
def crear_modelo_regularizado():
    modelo = keras.Sequential([
        layers.Input(shape=(28, 28)), layers.Flatten(),
        layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.30),
        layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.30),
        layers.Dense(10, activation="softmax")])
    modelo.compile(optimizer=keras.optimizers.Adam(0.001),
        loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

modelo_eval = crear_modelo_regularizado()
hist_eval = modelo_eval.fit(x_train, y_train, validation_data=(x_dev, y_dev),
    epochs=20, batch_size=64, callbacks=[callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True)], verbose=1)
print("Épocas ejecutadas:", len(hist_eval.history["loss"]))


In [ ]:
loss_test, accuracy_test = modelo_eval.evaluate(x_test, y_test, verbose=0)
probabilidades_test = modelo_eval.predict(x_test, verbose=0)
y_pred = np.argmax(probabilidades_test, axis=1)
precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
metricas = pd.DataFrame({"metrica": ["Loss", "Accuracy", "Precision macro", "Recall macro", "F1 macro"],
    "resultado": [loss_test, accuracy_test, precision_macro, recall_macro, f1_macro]})
display(metricas)


In [ ]:
print(classification_report(y_test, y_pred, digits=4, zero_division=0))
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(10)).plot(
    ax=ax, cmap="Blues", colorbar=False)
plt.title("Matriz de confusión - MNIST")
plt.show()


In [ ]:
cm_errores = cm.copy()
np.fill_diagonal(cm_errores, 0)
real_confundido, predicho_como = np.unravel_index(np.argmax(cm_errores), cm_errores.shape)
cantidad = cm_errores[real_confundido, predicho_como]
print(f"Confusión más frecuente: {real_confundido} como {predicho_como} ({cantidad} veces)")
indices = np.where((y_test == real_confundido) & (y_pred == predicho_como))[0]
if len(indices):
    idx = indices[0]
    plt.imshow(x_test[idx], cmap="gray")
    plt.title(f"Real: {y_test[idx]} | Predicción: {y_pred[idx]}")
    plt.axis("off"); plt.show()
